# Chạy LLM extractor (Qwen2.5-7B) trên Google Colab
Self-host ≤9B, KHÔNG API ngoài. Runtime → Change runtime type → **GPU** (T4/L4/A100).
Mẹo: cache model vào Google Drive để khỏi tải lại 15GB mỗi phiên.

In [1]:
# 1) (Khuyến nghị) Mount Drive + cache HuggingFace vào Drive để khỏi tải lại model
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'   # cache model bền vững
    print('HF_HOME =', os.environ['HF_HOME'])
except Exception as e:
    print('Bỏ qua Drive:', e)

Mounted at /content/drive
HF_HOME = /content/drive/MyDrive/hf_cache


In [2]:
# 2) Lấy code
%cd /content
![ -d viettel-medreason ] || git clone https://github.com/tienndat1904/viettel-medreason.git
%cd /content/viettel-medreason
!git pull -q

/content
Cloning into 'viettel-medreason'...
remote: Enumerating objects: 403, done.
remote: Counting objects: 100% (403/403), done.
remote: Compressing objects: 100% (263/263), done.
remote: Total 403 (delta 146), reused 370 (delta 121), pack-reused 0 (from 0)
Receiving objects: 100% (403/403), 3.23 MiB | 25.80 MiB/s, done.
Resolving deltas: 100% (146/146), done.
/content/viettel-medreason


In [3]:
# 3) Cài dependencies
!pip install -q -r requirements.txt
!pip install -q -r requirements-gpu.txt
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 9.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.8/177.8 kB 18.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: Cannot install -r requirements.txt (line 16) and transformers==4.46.3 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
# 4) SMOKE TEST 3 file trước (kiểm model + JSON parse; lần đầu tải model ~15GB)
import os, shutil
os.makedirs('smoke/input', exist_ok=True)
for n in [1, 5, 50]:
    shutil.copy(f'data/test/input/{n}.txt', f'smoke/input/{n}.txt')
!python src/pipeline.py --input smoke/input --output smoke/out --backend llm
!cat smoke/out/5.json

[linker] backend=lexical | icd=kb(98186) rxnorm=kb(47719) | synonyms=53 brands=4150
[pipeline] backend=llm | 3 file | out=smoke/out
2026-07-02 09:24:32.130987: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
tokenizer_config.json: 7.30kB [00:00, 27.4MB/s]
vocab.json: 2.78MB [00:00, 62.1MB/s]
merges.txt: 1.67MB [00:00, 69.1MB/s]
tokenizer.json: 7.03MB [00:00, 138MB/s]
config.json: 100% 663/663 [00:00<00:00, 6.47MB/s]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 1778, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", 

In [5]:
# 5) Chạy full 100 file (~1–3h với HF generate; dùng 3B để lặp nhanh nếu cần)
!python src/pipeline.py --input data/test/input --output output --backend llm

[linker] backend=lexical | icd=kb(98186) rxnorm=kb(47719) | synonyms=53 brands=4150
[pipeline] backend=llm | 100 file | out=output
2026-07-02 09:25:02.382689: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 1778, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_

In [6]:
# 6) Validate + đóng gói + tải về
!python scripts/package_submission.py --output output --input data/test/input --n 100
from google.colab import files; files.download('output.zip')

❌ KHÔNG đóng gói — có lỗi:
  - THIẾU 1.json
  - THIẾU 2.json
  - THIẾU 3.json
  - THIẾU 4.json
  - THIẾU 5.json
  - THIẾU 6.json
  - THIẾU 7.json
  - THIẾU 8.json
  - THIẾU 9.json
  - THIẾU 10.json
  - THIẾU 11.json
  - THIẾU 12.json
  - THIẾU 13.json
  - THIẾU 14.json
  - THIẾU 15.json
  - THIẾU 16.json
  - THIẾU 17.json
  - THIẾU 18.json
  - THIẾU 19.json
  - THIẾU 20.json
  - THIẾU 21.json
  - THIẾU 22.json
  - THIẾU 23.json
  - THIẾU 24.json
  - THIẾU 25.json
  - THIẾU 26.json
  - THIẾU 27.json
  - THIẾU 28.json
  - THIẾU 29.json
  - THIẾU 30.json
  - THIẾU 31.json
  - THIẾU 32.json
  - THIẾU 33.json
  - THIẾU 34.json
  - THIẾU 35.json
  - THIẾU 36.json
  - THIẾU 37.json
  - THIẾU 38.json
  - THIẾU 39.json
  - THIẾU 40.json
  - THIẾU 41.json
  - THIẾU 42.json
  - THIẾU 43.json
  - THIẾU 44.json
  - THIẾU 45.json
  - THIẾU 46.json
  - THIẾU 47.json
  - THIẾU 48.json
  - THIẾU 49.json
  - THIẾU 50.json


FileNotFoundError: Cannot find file: output.zip

In [ ]:
# 7) Đo trên dev gold
!python src/eval/scorer.py --pred output --gold data/dev/gold --mode overlap
!python src/eval/eval_linking.py